In [23]:
# İlk olarak gerekli kütüphaneleri import ediyoruz.
import gymnasium as gym
import numpy as np
from collections import deque
import random
import torch
from torch import nn
import torch.nn.functional as F

In [22]:
class DeepQNetworkModel(nn.Module):
    # Modeli initialize edip network layer tanımı yaparken alacağı değerleri belirledim. Örn: in_features...
    def __init__(self, in_features, hidden_layers, out_features):
        super().__init__()

        # Sonrasında input ve output layerlarını tanımladım ve almaları gereken parametreleri girdim.
        self.first_layer = nn.Linear(in_features, hidden_layers)

        self.second_layer = nn.Linear(hidden_layers, out_features)

    # Forward fonksiyonunda ilk olarak elimizdeki veri lineer olmadığı için ReLu Activation uyguladım ve outputu return ettim.
    def forward(self, x):
        x = F.relu(self.first_layer(x))

        x = F.relu(self.second_layer(x))
        
        return x

In [20]:
# Önceki aksiyonları takip edebilmemiz için bir replay buffer oluşturuyoruz ve bu sonuçları bir araya toplama işlemini deque ile yapıyoruz.
class ReplayBuffer():
    def __init__(self, maxlen):
        self.memory = deque([], maxlen=maxlen)

    
    # Her bir transition olduğunda bunu memorye ekliyoruz.
    def append(self, transition):
        self.memory.append(transition)


    # Agent bazı durumlarda random bir karar verebilsin diye random sample alıyoruz.
    def sample(self, sample_size):
        return random.sample(self.memory, sample_size)


    def __len__(self):
        return len(self.memory)

In [21]:
class DeepQLearning():
    # İleride kullanacağımız bazı parametreleri tanımlıyoruz.
    BATCH_SIZE = 32 # Replay bufferda aldığımız training verisinin büyüklüğü.

    BUFFER_SIZE = 1000 # Replay bufferın büyüklüğü.
    
    LR = 0.001 # Sistemin öğrenme oranı.
    
    DISCOUNT = 0.9 # Kullanacğımız discount oranı    
    
    SYNC_STEPS = 10 # Policy ve target networkler arasında dengeyi sağlayacak ve onları daha efektip olarak kullanabileceğiz.

    loss_function = nn.MSELoss() # Mean squared error kullanarak modelimizin başarısını ölçeceğiz.
    
    optimizer = None # Modeli optimize etmek için kullanacağız. Şimdilik None olacak, ileride değişecek.

    
    actions = ["Left", "Down", "Right", "Up"] # 0, 1, 2, 3  Sol, Aşağı, Sağ, Aşağı

    
    # Elimizdeki statei tensor olarak kullanabilmemiz için. 16 state var ve bunlardan hangisi mevcut state ise onun değeri 1 ve geri kalanlar 0 olacak.
    def state_to_tensor_represantation(self, state:int, number_of_states:int): 
        input_tensor = torch.zeros(number_of_states)
    
        input_tensor[state] = 1
    
        return input_tensor

    
    # Training için bir fonksiyon oluşturuyoruz ve parametrelerini belirliyoruz.
    def train(self, episodes, is_slippery):
        # Ortamı oluşturduk ve visualize edebilmek için env.reset() ve env.render() komutlarını yazıyoruz.
        env = gym.make("FrozenLake-v1", map_name="4x4", is_slippery=False, render_mode="human")
    
        env.reset()
    
        env.render()
    
        number_of_states = env.observation_space.n # State sayısı
    
        number_of_actions = env.action_space.n # Action sayısı


        EPS = 1 # EPS modelin ne kadar rastgele ilerleyeceğini belirleyecek. EPS = 1 ise %100 random anlamına geliyor.
    
        memory = ReplayBuffer(self.BUFFER_SIZE)


        # Modeli train etmek için policy ve test etmek için de bir target network oluşturuyoruz.
        policy_dqn = DeepQNetworkModel(in_features=number_of_states, hidden_layers=number_of_states, out_features=number_of_actions)

        target_dqn = DeepQNetworkModel(in_features=number_of_states, hidden_layers=number_of_states, out_features=number_of_actions)


        # Optimizerı SGD olarak belirledim.
        self.optimizer = torch.optim.SGD(policy_dqn.parameters(), lr=self.LR)


        # Episode başına reward takibi yapmak için.
        rewards_per_episode = np.zeros(episodes)


        # EPS değerini takip etmek için.
        epsilon_history = []


        # Yukarıda bahsettiğimiz policy ve target networkleri takip etmek için kullanacağız.
        step_count = 0
            

        for i in range(episodes):
            state = env.reset()[0]  # İlk state için initalize ediyoruz.

            terminated = False      # Agent kuyuya düşerse veya hedefe oluşursa True olacak.

            truncated = False       # Agent 200'den fazla action alırsa True olacak. 


            # Agent kuyuya düştü mü, hedefe ulaştı mı ya da 200 action aldı mı sorularının takibini yapıyoruz.
            while not terminated and not truncated:

                # EPS değerine göre random bir action seçiyoruz.
                if random.random() < EPS:
                    action = env.action_space.sample()
                
                else:
                    # En işe yarar action seçiliyor.        
                    with torch.no_grad():
                        action = policy_dqn(self.state_to_tensor_represantation(state=state, number_of_states=number_of_states)).argmax().item()

                # Seçtiğimiz action uygulanıyor.
                new_state, reward, terminated, truncated, _ = env.step(action)

                
                # Buffer'a action kaydediliyor.
                memory.append((state, action, new_state, reward, terminated)) 

                
                # Sıradaki state geçişi.
                state = new_state

                
                # Step counta da 1 ekliyoruz.
                step_count += 1

            
            # Reward 1 ise değerini 1 olarak giriyoruz.
            if reward == 1:
                rewards_per_episode[i] = 1

            
            # Yeterince training yaptık mı ve en az 1 reward toplayabildik mi diye bakıyoruz.
            if len(memory) > self.BATCH_SIZE and np.sum(rewards_per_episode) > 0:
                mini_batch = memory.sample(self.BATCH_SIZE)
            
                # Optimize etme işlemi. 
                self.optimize(mini_batch, policy_dqn, target_dqn)        

                # EPS değerini değiştirmeye başlıyoruz ki sonuçlarımız da farklı olsun.
                EPS = max(EPS - 1 / episodes, 0)

                epsilon_history.append(EPS)


                # Belli bir süre sonra policy network değerlerini aynı şekilde target networke de veriyoruz.
                if step_count > self.SYNC_STEPS:
                    target_dqn.load_state_dict(policy_dqn.state_dict())

                    step_count = 0


        # Sistemi kapatıyoruz.
        env.close()

        
    # Optimizasyon(yukarıda kullandığımız).
    def optimize(self, batch_size, policy_dqn, target_dqn):

        # Inputları alıyoryuz ve q değerlerini saklamak için 2 boş liste oluşturuyoruz.
        number_of_states = policy_dqn.first_layer.in_features

        current_q_list = []
        target_q_list = []

        for state, action, new_state, reward, terminated in batch_size:

            if terminated: 
                # Agent ya hedefe ulaştı ya da kuyuya düştü. Sırasıyla reward 1 ve 0.
                # Terminated True ise target q reward olarak belirlenmeli.
                target = torch.FloatTensor([reward])
            else:
                # Target q değerini hesaplamak için formülünü kullanıyoruz.
                with torch.no_grad():
                    target = torch.FloatTensor(
                        reward + self.DISCOUNT * target_dqn(self.state_to_tensor_represantation(state=new_state, number_of_states=number_of_states)).max())

            # Elimizdeki q değerleri.
            current_q = policy_dqn(self.state_to_tensor_represantation(state=state, number_of_states=number_of_states))
            current_q_list.append(current_q)

            # Hedef q değerleri
            target_q = target_dqn(self.state_to_tensor_represantation(state=state, number_of_states=number_of_states)) 
            
            target_q[action] = target
            target_q_list.append(target_q)
                
        # Loss function kullanarak kaybımızı ölçüyoruz.
        loss = self.loss_function(torch.stack(current_q_list), torch.stack(target_q_list))

        # Optimizasyon
        self.optimizer.zero_grad()
        loss.backward()
        self.optimizer.step()
    

    # Test fonksiyonu.
    def test(self, episodes, is_slippery):
        env = gym.make("FrozenLake-v1", map_name="4x4", is_slippery=False, render_mode="human")
        
        env.reset()
        
        env.render()
        
        number_of_states = env.observation_space.n
        
        number_of_actions = env.action_space.n


        # Test için kullanacağımız policy yüklendi.
        policy_dqn = DeepQNetworkModel(in_features=number_of_states, hidden_layers=number_of_states, out_features=number_of_actions) 

        policy_dqn.eval() # Test yapacağımız için eval moduna geçtik. 


        for i in range(episodes):
            state = env.reset()[0]

            terminated = False

            truncated = False        


            while not terminated and not truncated:  

                # Policy network yardımıyla best action seçimi yapıyoruz.
                with torch.no_grad():
                    action = policy_dqn(self.state_to_tensor_represantation(state=state, number_of_states=number_of_states)).argmax().item()


                # Execute ediyoruz.
                state, reward, terminated, truncated, _ = env.step(action)

        # Sistemi de son olarak kapatıyoruz.
        env.close()


# Sonucumuzu görmek için bir instance tanımlıyoruz. Train size %30 olarak belirlendi. Değişik bir değer de seçilebilir. 
frozen_lake = DeepQLearning()

frozen_lake.train(700, is_slippery=True)

frozen_lake.test(300, is_slippery=True)